# Regulatory capital (Basel II current exposure)

`Portfolio` only values the book. `PortfolioB2CEM` is that same portfolio plus a current-exposure capital model. The model is still the only capital regime in the library.

Capital is, in outline:

- group contracts into structures (one contract each, or netted when `optimise` is true)
- add mark-to-market to a Basel add-on that scales with asset class, time to settlement, and net-to-gross ratio
- keep only the positive replacement cost
- multiply by the counterparty weighting and the capital charge

There is no margin in this notebook. The next one posts collateral against the same book.

`reg_cap_factors` accepts `ir`, `fx`, `equity`, `precious`, and `commodity`. The factors are larger for commodities than for FX, and the choice is an input: the curve does not select it.


In [1]:
import math
import sys
from datetime import date
from pathlib import Path

_cwd = Path.cwd().resolve()
for _candidate in [_cwd, *_cwd.parents]:
    _notebooks = _candidate / "python" / "notebooks"
    if (_notebooks / "sjd_setup.py").is_file():
        if str(_notebooks) not in sys.path:
            sys.path.insert(0, str(_notebooks))
        break
    if (_candidate / "sjd_setup.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise RuntimeError("Could not find python/notebooks/sjd_setup.py from " + str(_cwd))

import sjd_setup

sjd = sjd_setup.load_sjd()
print(sjd.__file__)


E:\Code\sjd\python\sjd_runtime\sjd.pyd


In [2]:
market = sjd_setup.sample_market(sjd)
book = sjd_setup.sample_contracts(sjd, market["forward"], market)
environment = market["environment"]
print("anchor", market["anchor"])
print(
    "collar", market["collar_settlement"],
    "put", book["put_strike"],
    "call", book["call_strike"],
)
print("forward", market["forward_settlement"], "strike", book["forward_strike"])
print(
    "asian", market["asian_settlement"],
    "averaging days", len(market["averaging_dates"]),
    "strike", book["asian_strike"],
)


anchor 2026-09-16
collar 2026-12-17 put 17.1 call 18.16
forward 2027-03-16 strike 17.41
asian 2027-07-02 averaging days 20 strike 17.72


## Same book, two factor sets

Both portfolios use `optimise=True` and the same credit parameters (weight 1, capital charge 8%). Commodity add-on factors are higher than FX factors inside one year, so commodity capital is higher.


In [3]:
credit = sjd_setup.credit_parameters(sjd)
commodity = sjd.PortfolioB2CEM(
    book["contracts"],
    sjd.reg_cap_factors("commodity"),
    credit,
    True,
)
fx = sjd.PortfolioB2CEM(
    sjd_setup.sample_contracts(sjd, market["forward"], market)["contracts"],
    sjd.reg_cap_factors("fx"),
    credit,
    True,
)
for name, portfolio in (("commodity", commodity), ("fx", fx)):
    report = portfolio.validate_report(environment)
    assert report.ok, [issue.message for issue in report.issues]
    print(f"{name:<12} {portfolio.get_regulatory_capital(environment):16,.2f}")

assert commodity.get_regulatory_capital(environment) > fx.get_regulatory_capital(environment)


commodity          376,664.68
fx                  67,254.26


## Netting the collar

`optimise=False` leaves the put and the call in separate structures, so each leg contributes its own add-on. `optimise=True` can put them in one structure because they share a maturity and a settlement. The net effective volume is smaller, and so is the add-on.


In [4]:
collar = sjd_setup.sample_contracts(sjd, market["forward"], market)
separate = sjd.PortfolioB2CEM(
    collar["contracts"][:2],
    sjd.reg_cap_factors("commodity"),
    credit,
    False,
)
netted = sjd.PortfolioB2CEM(
    sjd_setup.sample_contracts(sjd, market["forward"], market)["contracts"][:2],
    sjd.reg_cap_factors("commodity"),
    credit,
    True,
)
for name, portfolio in (("separate structures", separate), ("netted collar", netted)):
    report = portfolio.validate_report(environment)
    assert report.ok, [issue.message for issue in report.issues]
    print(f"{name:<20} {portfolio.get_regulatory_capital(environment):16,.2f}")

assert netted.get_regulatory_capital(environment) <= separate.get_regulatory_capital(environment)


separate structures         72,554.97
netted collar               56,000.00
